In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
train_df = pd.read_csv('../input/house-prices-advanced-regression-techniques/train.csv')

# --- 1. Calculate and list columns with missing values ---

# --- Calculate and plot columns with missing values ---

# Calculate the percentage of missing values
missing_percentage = train_df.isnull().sum() / len(train_df) * 100

# Filter for columns that have missing values and sort them
cols_with_missing = missing_percentage[missing_percentage > 0].sort_values(ascending=False)

# Plot the results as a horizontal bar graph
plt.figure(figsize=(12, 10))
sns.barplot(x=cols_with_missing, y=cols_with_missing.index)
plt.title('Percentage of Missing Values in Each Column', fontsize=16)
plt.xlabel('Percentage (%)', fontsize=12)
plt.ylabel('Columns', fontsize=12)
plt.show()
# --- 2. Remove columns with >18% missing values that aren't 'special' ---

# Define the list of columns where NA has a specific, meaningful category
special_na_cols = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
    'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish', 
    'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType'
]

# Identify columns to drop based on your rule
cols_to_drop = cols_with_missing[
    (cols_with_missing > 18) & (~cols_with_missing.index.isin(special_na_cols))
].index

# Drop the columns
df_processed = train_df.drop(columns=cols_to_drop)

print("--- Column Removal ---")
if len(cols_to_drop) > 0:
    print(f"Removed the following columns: {list(cols_to_drop)}")
else:
    print("No columns met the criteria to be removed.")

print(f"\nOriginal number of columns: {train_df.shape[1]}")
print(f"Number of columns after removal: {df_processed.shape[1]}")

In [ ]:
# Part 1A: Fill NA where it's a specific category like 'None' or 0
# -----------------------------------------------------------------
# For these columns, NA means the house lacks the feature. We fill with 'None'.
for col in ('Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
            'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish', 
            'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature'):
    if col in df_processed.columns: # Check if column exists
        df_processed[col] = df_processed[col].fillna('None')

# For these numeric columns, NA likely means 0.
for col in ('GarageYrBlt', 'GarageArea', 'GarageCars', 'MasVnrArea', 'BsmtFinSF1',
            'BsmtFinSF2', 'BsmtUnfSF','TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath'):
    if col in df_processed.columns:
        df_processed[col] = df_processed[col].fillna(0)

# Part 1B: Fill remaining true missing values
# -------------------------------------------
# For the rest, fill with the most frequent value (mode) of the column.
for col in ('MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st', 
            'Exterior2nd', 'SaleType', 'Functional', 'Utilities', 'MasVnrType'):
     if col in df_processed.columns:
        df_processed[col] = df_processed[col].fillna(df_processed[col].mode()[0])

# LotFrontage is a special case: impute using the median of the neighborhood.
if 'LotFrontage' in df_processed.columns:
    df_processed['LotFrontage'] = df_processed.groupby('Neighborhood')['LotFrontage'].transform(
        lambda x: x.fillna(x.median())
    )

# --- Verification ---
# Check if any missing values are left
remaining_na = df_processed.isnull().sum().sum()
if remaining_na == 0:
    print("SUCCESS: All missing values have been handled.")
else:
    print(f"WARNING: There are still {remaining_na} missing values to address.")



# These columns are numeric but represent categories, so we convert them to strings
cols_to_convert = ['MSSubClass', 'YrSold', 'MoSold']

for col in cols_to_convert:
    if col in df_processed.columns: # Check if column exists before converting
        df_processed[col] = df_processed[col].astype(str)

print("\nConverted numeric codes (MSSubClass, YrSold, MoSold) to categorical type.")

In [ ]:
# We'll use the dataframe from the previous step where missing values were handled
df_final = df_processed.copy()

# --- Step 1: Find the Top 10 Correlated Numeric Features ---

# Calculate the correlation matrix for numeric columns only
numeric_cols = df_final.select_dtypes(include=np.number)
corrmat = numeric_cols.corr()

# Get the 10 features with the highest correlation to SalePrice
k = 11 # We ask for 11 to include SalePrice itself
top_corr_cols = corrmat.nlargest(k, 'SalePrice')['SalePrice'].index
top_corr_cols = top_corr_cols.drop('SalePrice') # Remove SalePrice from the list

print("--- Top 10 Features Correlated with SalePrice ---")
print(top_corr_cols.tolist())
print("\n" + "="*50 + "\n")


# --- Step 2: Log Transform Skewed Features from this Top 10 List ---

# Identify skewed features from our top 10 list (skewness > 0.75 is a common threshold)
skewed_features = df_final[top_corr_cols].skew().sort_values(ascending=False)
highly_skewed = skewed_features[skewed_features > 0.75]

print("--- Highly Skewed Features to be Transformed ---")
print(highly_skewed.index.tolist())
print("\n" + "="*50 + "\n")

# Apply log transformation (log1p handles 0 values safely)
for col in highly_skewed.index:
    df_final[col] = np.log1p(df_final[col])


# --- Step 3: Apply IQR Outlier Removal to the (Now Transformed) Top 10 ---

print("--- Applying Targeted Outlier Removal ---")
print(f"Original number of rows: {df_final.shape[0]}")

# Loop through ONLY the top 10 columns and remove outliers
for col in top_corr_cols:
    if col in df_final.columns:
        # Calculate Q1, Q3, and IQR
        Q1 = df_final[col].quantile(0.25)
        Q3 = df_final[col].quantile(0.75)
        IQR = Q3 - Q1
        
        # Define the outlier fences
        lower_fence = Q1 - 1.5 * IQR
        upper_fence = Q3 + 1.5 * IQR
        
        # Filter the dataframe
        df_final = df_final[
            (df_final[col] >= lower_fence) & (df_final[col] <= upper_fence)
        ]

print(f"Number of rows after targeted removal: {df_final.shape[0]}")
rows_removed = df_processed.shape[0] - df_final.shape[0]
print(f"Total rows removed: {rows_removed}")

In [ ]:
# Set up the plotting area
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Box plot of the original data (with outliers)
sns.boxplot(y=df_processed['GrLivArea'], ax=ax[0])
ax[0].set_title('Original GrLivArea (With Outliers)')
ax[0].set_ylabel('Above Ground Living Area (sq ft)')

# Box plot of the cleaned data (without outliers)
sns.boxplot(y=df_final['GrLivArea'], ax=ax[1])
ax[1].set_title('Cleaned GrLivArea (Without Outliers)')
ax[1].set_ylabel('Above Ground Living Area (sq ft)')

plt.show()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# We will continue with the 'df_final' DataFrame from the previous step
df_normalized = df_final.copy()

# --- Identify the numeric columns to scale ---
# We exclude the target variable 'SalePrice' and its log-transformed version.
numeric_cols = df_normalized.select_dtypes(include=np.number).columns
cols_to_scale = [col for col in numeric_cols if col not in ['SalePrice', 'SalePrice_log']]

# --- Apply Normalization ---
scaler = MinMaxScaler()

# Fit the scaler to the data and transform the columns
df_normalized[cols_to_scale] = scaler.fit_transform(df_normalized[cols_to_scale])

print("--- DataFrame after Normalization ---")
# Display the first few rows to show the scaled values (now between 0 and 1)
print(df_normalized[cols_to_scale].head())

In [ ]:
# Set up the plotting area
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Original distribution (from the previously cleaned data)
sns.histplot(df_final['GrLivArea'], ax=ax[0], kde=True)
ax[0].set_title('GrLivArea Distribution Before Normalization')

# Normalized distribution
sns.histplot(df_normalized['GrLivArea'], ax=ax[1], kde=True)
ax[1].set_title('GrLivArea Distribution After Normalization')
plt.show()

In [ ]:
# Start with the fully scaled and cleaned dataframe from Part 4
df_encoded = df_normalized.copy()

# Define the mapping for quality scores. Note 'None' is 0.
quality_mapping = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5, 'None': 0}

# List of all ordinal columns that use this quality scale
ordinal_cols = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
    'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC'
]

# Apply the mapping to each ordinal column
for col in ordinal_cols:
    if col in df_encoded.columns: # Check if column exists
        df_encoded[col] = df_encoded[col].replace(quality_mapping)

print("--- Manually encoded ordinal features ---")
print(df_encoded[ordinal_cols].head())


# Now, apply one-hot encoding to the rest of the categorical columns
df_encoded = pd.get_dummies(df_encoded, drop_first=True)

print("\n--- Final DataFrame after both encoding steps ---")
print(f"Total number of columns: {df_encoded.shape[1]}")
print(df_encoded.head())